# Task 3 — Step 2: GRPO Equal-Generation Group-Size Study ($K \in \{2, 4, 8\}$)

### Research Question
Holding the **total generation budget constant** (equal total completions sampled), how does group size $K$ alter:
1. **Informative-group rate**: Fraction of prompt groups with non-zero standard deviation $\sigma_r > 0$?
2. **Variance of relative advantages**: Signal-to-noise ratio of within-group standardization?
3. **Prompt difficulty interaction**: Do easy or hard prompts benefit more from larger $K$?


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Offline Group-Size Regrouping on Cached Rollouts


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task3_grpo.analyze_group_size --config configs/grpo.yaml
else:
    print("[OK] Loading saved group-size analysis...")


## 2. Summary Table across Group Sizes $K$


In [ ]:
group_file = "results/task3_grpo/grpo_group_size_analysis.json"
if os.path.exists(group_file):
    with open(group_file, encoding='utf-8') as f:
        g_data = json.load(f)
        
    rows = []
    for k_str in sorted(g_data.keys(), key=int):
        st = g_data[k_str]
        diff = st.get("difficulty_breakdown", {})
        rows.append({
            "Group Size (K)": int(k_str),
            "Total Groups": st.get("total_groups", 0),
            "Total Generations": st.get("total_generations", 0),
            "Informative Group %": f"{st.get('informative_group_fraction', 0)*100:.1f}%",
            "Within-Group Std": f"{st.get('mean_within_group_reward_std', 0):.4f}",
            "Advantage Variance": f"{st.get('relative_signal_variance', 0):.4f}",
            "Easy Inf %": f"{diff.get('easy', {}).get('informative_fraction', 0)*100:.1f}%",
            "Hard Inf %": f"{diff.get('hard', {}).get('informative_fraction', 0)*100:.1f}%"
        })
    df_k = pd.DataFrame(rows)
    display(df_k)


## 3. Visualizing Group Informativeness Across Difficulty Tiers


In [ ]:
if os.path.exists(group_file):
    ks = [r["Group Size (K)"] for r in rows]
    overall = [float(r["Informative Group %"].replace('%','')) for r in rows]
    easy = [float(r["Easy Inf %"].replace('%','')) for r in rows]
    hard = [float(r["Hard Inf %"].replace('%','')) for r in rows]
    
    x = np.arange(len(ks))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - width, overall, width, label='Overall Dataset', color='#3B82F6', alpha=0.85)
    ax.bar(x, easy, width, label='Easy Prompts', color='#10B981', alpha=0.85)
    ax.bar(x + width, hard, width, label='Hard Prompts', color='#EF4444', alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels([f"K = {k}" for k in ks])
    ax.set_ylabel('Informative Groups (%)')
    ax.set_title('Group Informativeness (σ_r > 0) by Difficulty and Group Size')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## 4. Publication Figure Inspection


In [ ]:
p = "report/figures/task3_grpo_group_size_study.png"
if os.path.exists(p):
    print(f"Displaying publication figure: {p}")
    display(Image(filename=p, width=650))


## 5. Research Question 1 Analysis & Takeaways
- Why do smaller groups ($K=2$) have higher zero-variance rates on difficult prompts?
- What is the trade-off between spending compute on more prompts vs. more completions per prompt?
